# Problem 5 — The Tide Ledger

This 14-part arc follows one fresh harbor-town corpus from tokens to a compressed similarity model. Run the notebook in order: every later part uses names established earlier.


In [ ]:
import os
import pathlib

_root = next(
    path for path in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
os.environ["GENSIM_DATA_DIR"] = str(_root / "reference" / "cache" / "gensim")

import gensim.downloader
import numpy as np
from gensim.utils import simple_preprocess

corpus_path = _root / "mocktests" / "r1-001" / "data" / "corpus.txt"
corpus = corpus_path.read_text(encoding="utf-8")
kv = gensim.downloader.load("glove-wiki-gigaword-100")
print(f"Loaded {len(corpus.split())} whitespace-separated words from {corpus_path.name}.")


## Part 5.1 (5 points)

**Type:** programming · **Difficulty: intro** · **Answer form: code** · **Concepts: tokenization**  
**Flag:** Coding allowed.

Tokenize the supplied corpus with gensim.utils.simple_preprocess. Store the list in tokens, the number of token occurrences in token_count, and the number of distinct token types in type_count. Print both counts.


In [ ]:
tokens = simple_preprocess(corpus)
token_count = len(tokens)
type_count = len(set(tokens))
print("token count:", token_count)
print("type count:", type_count)

In [ ]:
ANSWER = 311
assert np.isclose(ANSWER, token_count, atol=0, rtol=0)

## Part 5.2 (5 points)

**Type:** theory · **Difficulty: intro** · **Answer form: short-answer** · **Concepts: tokenization**  
**Flag:** Coding not required.

A classmate runs the supplied cell below in two fresh Python processes and sees the same counts but a different preview order. Explain why the two census prints can differ after list(set(tokens)). State one other piece of information that conversion to a set loses.


In [ ]:
deduplicated_with_set = list(set(tokens))
print("set census:", len(deduplicated_with_set), deduplicated_with_set[:8])


**Solution.** Python sets are unordered collections whose iteration order depends on the
process-specific hash layout, so `list(set(tokens))` need not have the same preview order in
fresh processes. Converting the token sequence to a set also loses multiplicity (token
frequencies), as well as the corpus order and positions.

In [ ]:
ANSWER = "hash-dependent set order; multiplicity and sequence order are lost"
assert len(tokens) != len(set(tokens))
assert token_count != type_count
assert any(tokens.count(token) != deduplicated_with_set.count(token) for token in set(tokens))

## Part 5.3 (5 points)

**Type:** programming · **Answer form: code**  
**Flag:** Coding allowed.

Consume tokens from Part 5.1. Preserve first-occurrence order while removing duplicates, filter the result to words in kv.key_to_index, and store it as embedded_tokens. Look up every retained vector and store the vectors in embedded_vectors. At each lookup, apply the course boundary cast np.asarray(..., dtype=np.float64). Construct oov_tokens even though it may be empty; the filter code itself is graded by inspection.

*Instructor metadata — Difficulty: core · Concepts: gensim-usage, word-embeddings*


In [ ]:
ordered_types = list(dict.fromkeys(tokens))
embedded_tokens = [token for token in ordered_types if token in kv.key_to_index]
oov_tokens = [token for token in ordered_types if token not in kv.key_to_index]
embedded_vectors = [np.asarray(kv[token], dtype=np.float64) for token in embedded_tokens]
print("embedded types:", len(embedded_tokens), "| OOV types:", len(oov_tokens))

In [ ]:
ANSWER = 220
assert np.isclose(ANSWER, len(embedded_tokens), atol=0, rtol=0)

## Part 5.4 (5 points)

**Type:** programming · **Difficulty: intro** · **Answer form: code** · **Concepts: embedding-matrices, numpy-arrays**  
**Flag:** Coding allowed.

Stack embedded_vectors from Part 5.3 into the float64 matrix W_raw, with one retained token per row and exactly 100 columns. The identifier and orientation are required. Assert its shape and dtype.


In [ ]:
W_raw = np.stack(embedded_vectors, axis=0)
N = len(embedded_tokens)
assert W_raw.shape == (N, 100)
assert W_raw.dtype == np.float64

In [ ]:
ANSWER = 220
assert np.isclose(ANSWER, W_raw.shape[0], atol=0, rtol=0)

## Part 5.5 (5 points)

**Type:** programming · **Difficulty: core** · **Answer form: code** · **Concepts: broadcasting, vectorization**  
**Flag:** Coding allowed.  
**Ban (zero points for this part):** Do not use loops or any np.linalg function.

Consume W_raw. Compute each row's squared length in row_sq, its length in row_norms with shape (N, 1), and the unit-row matrix W by broadcasting. Keep all three required identifiers.


In [ ]:
row_sq = (W_raw * W_raw).sum(axis=1)
row_norms = np.sqrt(row_sq)[:, None]
W = W_raw / row_norms
assert row_sq.shape == (N,)
assert row_norms.shape == (N, 1)
assert W.shape == W_raw.shape
assert np.allclose((W * W).sum(axis=1), 1.0, atol=1e-12, rtol=0)

In [ ]:
unit_row_count = int(np.isclose((W * W).sum(axis=1), 1.0, atol=1e-12, rtol=0).sum())
ANSWER = 220
assert np.isclose(ANSWER, unit_row_count, atol=0, rtol=0)

## Part 5.6 (5 points)

**Type:** theory · **Difficulty: core** · **Answer form: short-answer** · **Concepts: cosine-similarity, unit-vectors**  
**Flag:** Coding not required.

For any two rows of the unit-row matrix W from Part 5.5, give the exact range of their dot product. State precisely when each endpoint occurs.


**Solution.** The exact range is $[-1,1]$. By equality in Cauchy--Schwarz for unit
vectors, the dot product is $1$ exactly when the two rows are identical and is $-1$ exactly
when one row is the negative of the other.

In [ ]:
ANSWER = "[-1, 1]; -1 for opposite unit rows, 1 for identical unit rows"
S = W @ W.T
off_diagonal = S[~np.eye(N, dtype=bool)]
assert off_diagonal.min() >= -1.0
assert off_diagonal.max() <= 1.0
assert np.unique(W, axis=0).shape[0] == N
assert off_diagonal.min() > -1.0
assert off_diagonal.max() < 1.0

## Part 5.7 (15 points)

**Type:** theory · **Answer form: proof**  
**Flags:** Reasoning required; coding not allowed.

Define the fresh corpus similarity matrix by $S = WW^T$, using Part 5.5's rows $w_1$ through $w_N$. Express $S_{ij}$ entrywise as a sum over the 100 embedding coordinates. Then prove from that expression that $S$ is symmetric and every diagonal entry equals $1$. Cite the relevant property of $W$ at each step.

*Instructor metadata — Difficulty: advanced · Concepts: similarity-matrices, matrix-multiplication*


**Proof.** Write row $i$ of $W$ as
$w_i=(w_{i1},\ldots,w_{i,100})$. Matrix multiplication gives

$$S_{ij}=(WW^T)_{ij}=\sum_{k=1}^{100}W_{ik}(W^T)_{kj}
=\sum_{k=1}^{100}w_{ik}w_{jk}.$$

Because real scalar multiplication commutes,
$S_{ji}=\sum_k w_{jk}w_{ik}=\sum_k w_{ik}w_{jk}=S_{ij}$, so $S=S^T$.
This uses only that both entries come from the same real matrix $W$. On the diagonal,
$S_{ii}=\sum_k w_{ik}^2=w_i\cdot w_i=\lVert w_i\rVert_2^2=1$ because Part 5.5
normalized every row of $W$ to unit length. Thus $S$ is symmetric and has an all-ones
diagonal. $\square$

*Instructor metadata — Difficulty: core · Concepts: broadcasting, vectorization*


In [ ]:
# Supplied for later parts after you complete the proof.
S = W @ W.T
assert S.shape == (N, N)


In [ ]:
proof_invariant = "S is symmetric and every diagonal entry is 1"
ANSWER = "S is symmetric and every diagonal entry is 1"
assert ANSWER == proof_invariant
assert np.allclose(S, S.T, atol=1e-12, rtol=0)
assert np.allclose(np.diag(S), 1.0, atol=1e-12, rtol=0)

## Part 5.8 (5 points)

**Type:** programming · **Difficulty: core** · **Answer form: code** · **Concepts: svd**  
**Flag:** Coding allowed; np.linalg.svd is explicitly allowed in this part.

Compute the thin SVD of Part 5.5's W with full_matrices=False, using U_thin, sigma, and Vt_thin. Report sigma and assert that it is non-increasing.


In [ ]:
U_thin, sigma, Vt_thin = np.linalg.svd(W, full_matrices=False)
print("descending singular values:", sigma)
assert np.all(np.diff(sigma) <= 1e-12)

In [ ]:
ANSWER = 100
assert np.isclose(ANSWER, sigma.size, atol=0, rtol=0)

## Part 5.9 (5 points)

**Type:** theory · **Difficulty: core** · **Answer form: short-answer** · **Concepts: svd, singular-values**  
**Flag:** Coding not required.

Let q = min(N, 100). For this exact (N, 100) matrix, state the shapes returned by thin SVD and by full SVD: U, sigma, and Vt in each case. Explain which output contains the q singular values of W, and which additional spectrum is needed for the N by N matrix S.


**Solution.** Here $N=220$ and $q=\min(N,100)=100$. Thin SVD returns
$U\in\mathbb{R}^{220\times100}$, `sigma.shape == (100,)`, and
$V^T\in\mathbb{R}^{100\times100}$. Full SVD returns
$U\in\mathbb{R}^{220\times220}$, the same 100-entry `sigma`, and
$V^T\in\mathbb{R}^{100\times100}$. In either call, `sigma` contains the $q$ singular
values of $W$. The $220\times220$ matrix $S=WW^T$ additionally has $N-q=120$ zero
eigenvalues; they must be appended to the squared singular values to obtain its full spectrum.

In [ ]:
ANSWER = "thin: (220,100),(100,),(100,100); full: (220,220),(100,),(100,100); S adds 120 zeros"
U_full, sigma_full, Vt_full = np.linalg.svd(W, full_matrices=True)
assert (U_thin.shape, sigma.shape, Vt_thin.shape) == ((220, 100), (100,), (100, 100))
assert (U_full.shape, sigma_full.shape, Vt_full.shape) == ((220, 220), (100,), (100, 100))
assert N - sigma.size == 120

## Part 5.10 (5 points)

**Type:** programming · **Difficulty: advanced** · **Answer form: code** · **Concepts: spectral-decomposition, svd**  
**Flag:** Coding allowed.

Consume W, S, and sigma. Compute the full SVD of W as U_full, sigma_full, Vt_full. Build lambda_full, the length-N eigenvalue vector for S, by squaring the singular values and padding all remaining entries with zeros. Then reconstruct S as S_spectral = U_full @ diag(lambda_full) @ U_full.T and verify the decomposition.


In [ ]:
U_full, sigma_full, Vt_full = np.linalg.svd(W, full_matrices=True)
assert np.allclose(sigma_full, sigma, atol=1e-12, rtol=0)
lambda_full = np.pad(sigma**2, (0, N - sigma.size))
S_spectral = U_full @ np.diag(lambda_full) @ U_full.T
assert U_full.shape == (N, N)
assert lambda_full.shape == (N,)
assert np.allclose(S_spectral, S, atol=1e-10, rtol=0)

In [ ]:
ANSWER = 220.0
assert np.isclose(ANSWER, lambda_full.sum(), atol=1e-10, rtol=0)

## Part 5.11 (15 points)

**Type:** theory · **Answer form: proof**  
**Flags:** Reasoning required; coding not allowed.

Using Part 5.10's spectral decomposition, let $S_r$ retain the first $r$ eigenpairs of $S$. Derive a formula for the relative squared Frobenius error $\lVert S-S_r\rVert_F^2 / \lVert S\rVert_F^2$ using only the singular values $\sigma$ of $W$. Your proof must justify why fourth powers appear and identify the numerator as a tail sum.

*Instructor metadata — Difficulty: advanced · Concepts: low-rank-approximation, frobenius-norm*


**Proof.** From Part 5.10, $S=U\,\mathrm{diag}(\lambda_1,\ldots,
\lambda_N)U^T$ with orthonormal columns $u_k$, where
$\lambda_k=\sigma_k^2$ for $1\le k\le q$ and $\lambda_k=0$ afterward. Hence

$$S-S_r=\sum_{k=r+1}^{N}\lambda_k u_k u_k^T.$$

The matrices $u_k u_k^T$ are orthonormal under the Frobenius inner product because
$\langle u_i u_i^T,u_j u_j^T\rangle_F=(u_i^T u_j)^2$. Pythagoras therefore gives

$$\lVert S-S_r\rVert_F^2=\sum_{k=r+1}^{N}\lambda_k^2
=\sum_{k=r+1}^{q}(\sigma_k^2)^2=\sum_{k=r+1}^{q}\sigma_k^4.$$

Similarly, $\lVert S\rVert_F^2=\sum_{k=1}^{q}\sigma_k^4$. The fourth powers appear
because an eigenvalue of $S$ is already the square of a singular value of $W$, and the
squared Frobenius norm squares that eigenvalue once more. Therefore

$$\frac{\lVert S-S_r\rVert_F^2}{\lVert S\rVert_F^2}
=\frac{\sum_{k=r+1}^{q}\sigma_k^4}{\sum_{k=1}^{q}\sigma_k^4},$$

whose numerator is exactly the spectral-energy tail omitted after rank $r$. $\square$

*Instructor metadata — Difficulty: advanced · Concepts: spectral-decomposition, svd*


In [ ]:
ANSWER = "sum(sigma[r:]**4) / sum(sigma**4)"
for r in range(1, 5):
    S_r = U_full[:, :r] @ np.diag(sigma[:r] ** 2) @ U_full[:, :r].T
    tail_ratio = (sigma[r:] ** 4).sum() / (sigma ** 4).sum()
    reconstruction_ratio = ((S - S_r) ** 2).sum() / (S ** 2).sum()
    assert np.isclose(tail_ratio, reconstruction_ratio, atol=1e-12, rtol=0)

## Part 5.12 (5 points)

**Type:** programming · **Difficulty: core** · **Answer form: code** · **Concepts: low-rank-approximation, vectorization**  
**Flag:** Coding allowed.  
**Constraint:** Build no truncated matrix or factorization.

Consume Part 5.8's descending sigma and Part 5.11's identity. With exactly one np.cumsum call, compute rel_sq_errors, where entry r - 1 is the relative squared Frobenius error of S_r for every rank r = 1, ..., q.


In [ ]:
spectral_energy = sigma**4
rel_sq_errors = 1.0 - np.cumsum(spectral_energy) / spectral_energy.sum()
assert rel_sq_errors.shape == sigma.shape
assert np.all(np.diff(rel_sq_errors) <= 1e-12)

In [ ]:
ANSWER = 100
assert np.isclose(ANSWER, rel_sq_errors.size, atol=0, rtol=0)

## Part 5.13 (5 points)

**Type:** programming · **Difficulty: core** · **Answer form: code** · **Concepts: low-rank-approximation, aggregation-axis**  
**Flag:** Coding allowed.

Using Part 5.12's rel_sq_errors and error_budget = 0.08, find the smallest positive integer r_star whose error is at most the budget. Do not recompute an SVD. Keep the two supplied certificate asserts: they must prove feasibility and minimality.


In [ ]:
error_budget = 0.08
r_star = int(np.flatnonzero(rel_sq_errors <= error_budget)[0] + 1)
assert rel_sq_errors[r_star - 1] <= error_budget
assert r_star == 1 or rel_sq_errors[r_star - 2] > error_budget
print("smallest feasible rank:", r_star)

In [ ]:
ANSWER = 3
assert np.isclose(ANSWER, r_star, atol=0, rtol=0)

## Part 5.14 (5 points)

**Type:** theory · **Difficulty: core** · **Answer form: short-answer** · **Concepts: low-rank-approximation**  
**Flag:** Coding not required.

Consume N and r_star. If the symmetric rank-r_star approximation is stored as U_full[:, :r_star] plus its r_star retained eigenvalues, how many floating-point numbers are stored, versus storing all of S? Give the general crossover inequality in N and r under which the factorization uses fewer numbers, then state whether Part 5.13's selected factorization is smaller.


**Solution.** Storing $U_{[:,1:r]}$ and the $r$ eigenvalues uses $Nr+r=r(N+1)$
floating-point numbers, versus $N^2$ for dense $S$. The factorization is smaller exactly
when $r(N+1)<N^2$. Here $N=220$ and $r_\star=3$, so it stores
$3(220+1)=663$ numbers versus $220^2=48{,}400$; it is smaller.

In [ ]:
stored_scalars = r_star * (N + 1)
ANSWER = 663
assert np.isclose(ANSWER, stored_scalars, atol=0, rtol=0)